# RS3 sequence model: fixed split, 100 Optuna trials

Run this notebook from the repository's `code` directory.

It follows the repository's own data-loading and feature-generation logic:

- loads datasets named in `../data/processed/train_data_names.csv`;
- uses `dataset_list` from `datasets.py`;
- uses `get_feature_df()` from `core.py`;
- creates one fixed grouped and dataset-stratified split with seed 42;
- uses approximately 64% training, 16% validation, and 20% unseen data;
- runs 100 Optuna trials;
- records only:
  - validation MSE;
  - unseen Pearson correlation;
  - unseen Spearman correlation.

The unseen metrics are calculated for every trial as requested, but Optuna selects trials using validation MSE only.


In [1]:
from pathlib import Path
import json
import warnings

import joblib
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold

from datasets import dataset_list
from core import get_feature_df

print("LightGBM version:", lgb.__version__)
print("Optuna version:", optuna.__version__)


/opt/anaconda3/envs/rs_dev_venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LightGBM version: 4.6.0
Optuna version: 4.9.0


## Configuration

Because this notebook is stored in `rs_dev/code`, repository paths begin with `../`.

The original RS3 notebook uses `learning_rate=0.01`, `n_estimators=5000`, and tunes `num_leaves` and `min_child_samples`. This notebook retains that setup.


In [3]:
# Reproducibility
SPLIT_SEED = 42
OPTUNA_SEED = 42
MODEL_SEED = 42

# Experiment
N_TRIALS = 100
EARLY_STOPPING_ROUNDS = 20

# Paths relative to rs_dev/code
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../models/rs3_fixed_split_100_trials")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_NAMES_FILE = PROCESSED_DIR / "train_data_names.csv"

assert TRAIN_NAMES_FILE.exists(), f"Missing file: {TRAIN_NAMES_FILE.resolve()}"

print("Output directory:", OUTPUT_DIR.resolve())


Output directory: /Users/zhangjiongyu/rs_dev/models/rs3_fixed_split_100_trials


## Load and harmonize the original RS3 training datasets

The duplicate `Doench2016` line in `train_data_names.csv` does **not** load the dataset twice here. The repository iterates once through `dataset_list` and checks whether each dataset name is present in the list.


In [5]:
train_data_names = pd.read_csv(TRAIN_NAMES_FILE)["name"].dropna().astype(str).tolist()

print("Names listed in train_data_names.csv:")
print(train_data_names)

train_data_list = [ds for ds in dataset_list if ds.name in train_data_names]

print("\nDatasets found in dataset_list:")
print([ds.name for ds in train_data_list])

missing_names = sorted(set(train_data_names) - {ds.name for ds in train_data_list})
if missing_names:
    print("\nWarning: names not found in dataset_list:", missing_names)

for ds in train_data_list:
    ds.load_data()
    ds.set_sgrnas()


Names listed in train_data_names.csv:
['Kim2019_train', 'Doench2014_mouse', 'Doench2014_human', 'Doench2016', 'Wang2014', 'Xiang2021', 'Munoz2016', 'Doench2016']

Datasets found in dataset_list:
['Doench2014_mouse', 'Doench2014_human', 'Doench2016', 'Kim2019_train', 'Wang2014', 'Xiang2021', 'Munoz2016']


In [7]:
sg_df_list = []

for ds in train_data_list:
    sg_df = ds.get_sg_df(include_group=True, include_activity=True).copy()
    sg_df["dataset"] = ds.name
    sg_df["tracr"] = ds.tracr
    sg_df_list.append(sg_df)

# Reproduce the target-group construction used in the repository.
sg_df_groups = (
    pd.concat(sg_df_list, ignore_index=True)
    .groupby("sgRNA Context Sequence", as_index=False)
    .agg(
        n_conditions=("sgRNA Context Sequence", "count"),
        target=(
            "sgRNA Target",
            lambda x: ", ".join(
                sorted({
                    str(s).upper()
                    for s in x
                    if not pd.isna(s) and str(s).strip() != ""
                })
            ),
        ),
    )
)

# Guides without an annotated target become singleton groups.
sg_df_groups["target"] = sg_df_groups.apply(
    lambda row: (
        row["target"]
        if row["target"] != ""
        else row["sgRNA Context Sequence"]
    ),
    axis=1,
)

all_data = (
    pd.concat(sg_df_list, ignore_index=True)
    .merge(
        sg_df_groups[["sgRNA Context Sequence", "target"]],
        how="inner",
        on="sgRNA Context Sequence",
    )
    .sort_values(["dataset", "target"])
    .reset_index(drop=True)
)

required_columns = {
    "sgRNA Context Sequence",
    "sgRNA Activity",
    "dataset",
    "tracr",
    "target",
}
missing_columns = required_columns - set(all_data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

all_data["sgRNA Activity"] = pd.to_numeric(
    all_data["sgRNA Activity"], errors="coerce"
)
all_data = all_data.dropna(
    subset=[
        "sgRNA Context Sequence",
        "sgRNA Activity",
        "dataset",
        "tracr",
        "target",
    ]
).reset_index(drop=True)

print("Combined shape:", all_data.shape)
print("Unique context sequences:", all_data["sgRNA Context Sequence"].nunique())
print("Unique target groups:", all_data["target"].nunique())
print("\nRows per dataset:")
display(all_data["dataset"].value_counts().rename("n").to_frame())
print("\nRows per tracrRNA:")
display(all_data["tracr"].value_counts().rename("n").to_frame())


Combined shape: (51087, 8)
Unique context sequences: 46526
Unique target groups: 20637

Rows per dataset:


,n
dataset,
Munoz2016,21136
Kim2019_train,12832
Xiang2021,11397
Doench2016,2536
Doench2014_mouse,1169
Wang2014,1022
Doench2014_human,995



Rows per tracrRNA:


,n
tracr,
Hsu2013,29951
Chen2013,21136


## Create one fixed 64%/16%/20% split

Two nested `StratifiedGroupKFold(n_splits=5)` operations are used:

1. Outer fold: 80% seen and 20% unseen.
2. Inner fold within seen data: 80% training and 20% validation.

Groups are based on `target`, so the same target group cannot occur in multiple subsets. Stratification uses `dataset`, matching the repository's grouped cross-validation principle.


In [9]:
# Outer fixed split: seen versus unseen
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SPLIT_SEED,
)

seen_idx, unseen_idx = next(
    outer_splitter.split(
        X=all_data,
        y=all_data["dataset"],
        groups=all_data["target"],
    )
)

seen_data = all_data.iloc[seen_idx].reset_index(drop=True)
unseen_data = all_data.iloc[unseen_idx].reset_index(drop=True)

# Inner fixed split: training versus validation within seen data
inner_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SPLIT_SEED,
)

train_idx, validation_idx = next(
    inner_splitter.split(
        X=seen_data,
        y=seen_data["dataset"],
        groups=seen_data["target"],
    )
)

train_data = seen_data.iloc[train_idx].reset_index(drop=True)
validation_data = seen_data.iloc[validation_idx].reset_index(drop=True)

# Leakage checks
train_targets = set(train_data["target"])
validation_targets = set(validation_data["target"])
unseen_targets = set(unseen_data["target"])

assert train_targets.isdisjoint(validation_targets)
assert train_targets.isdisjoint(unseen_targets)
assert validation_targets.isdisjoint(unseen_targets)

split_summary = pd.DataFrame(
    {
        "subset": ["train", "validation", "unseen"],
        "n_rows": [len(train_data), len(validation_data), len(unseen_data)],
        "fraction": [
            len(train_data) / len(all_data),
            len(validation_data) / len(all_data),
            len(unseen_data) / len(all_data),
        ],
        "n_target_groups": [
            train_data["target"].nunique(),
            validation_data["target"].nunique(),
            unseen_data["target"].nunique(),
        ],
    }
)

display(split_summary)

print("Dataset distribution by subset:")
display(
    pd.concat(
        {
            "train": train_data["dataset"].value_counts(normalize=True),
            "validation": validation_data["dataset"].value_counts(normalize=True),
            "unseen": unseen_data["dataset"].value_counts(normalize=True),
        },
        axis=1,
    ).fillna(0)
)


,subset,n_rows,fraction,n_target_groups
0,train,32563,0.637403,13002
1,validation,8498,0.166344,3470
2,unseen,10026,0.196253,4165


Dataset distribution by subset:


,train,validation,unseen
dataset,,,
Munoz2016,0.440868,0.437279,0.305605
Kim2019_train,0.241593,0.233231,0.297526
Xiang2021,0.219697,0.227348,0.230501
Doench2016,0.049504,0.000000,0.092160
Doench2014_mouse,0.018395,0.057072,0.008478
Wang2014,0.018395,0.025418,0.020646
Doench2014_human,0.011547,0.019652,0.045083


## Generate RS3 sequence features

`get_feature_df()` calls `sglearn.featurize_guides()` and adds tracrRNA indicator features, which is the repository's sequence-feature pipeline.


In [11]:
X_train = get_feature_df(train_data)
X_validation = get_feature_df(validation_data)
X_unseen = get_feature_df(unseen_data)

# Force identical feature order.
X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0)
X_unseen = X_unseen.reindex(columns=X_train.columns, fill_value=0)

y_train = train_data["sgRNA Activity"].to_numpy(dtype=float)
y_validation = validation_data["sgRNA Activity"].to_numpy(dtype=float)
y_unseen = unseen_data["sgRNA Activity"].to_numpy(dtype=float)

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_unseen:", X_unseen.shape)

assert list(X_train.columns) == list(X_validation.columns)
assert list(X_train.columns) == list(X_unseen.columns)


100%|███████████████████████████████████| 10026/10026 [00:01<00:00, 6013.07it/s]


X_train: (32563, 632)
X_validation: (8498, 632)
X_unseen: (10026, 632)


## Metric helpers


In [15]:
def safe_pearson(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(pearsonr(y_true, y_pred)[0])


def safe_spearman(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(y_true, y_pred)[0])


## Run 100 fixed-split Optuna trials

Only two hyperparameters are varied, matching the repository's original optimization:

- `num_leaves`: 8–256
- `min_child_samples`: 8–256

All trials use the exact same training, validation, and unseen subsets. Optuna minimizes validation MSE only.


In [17]:
trial_records = []
trial_models = {}


def objective(trial):
    num_leaves = trial.suggest_int("num_leaves", 8, 256)
    min_child_samples = trial.suggest_int("min_child_samples", 8, 256)

    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=MODEL_SEED,
        n_jobs=-1,
        learning_rate=0.01,
        n_estimators=5000,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_validation, y_validation)],
        eval_metric="mse",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose=False,
            )
        ],
    )

    best_iteration = (
        model.best_iteration_
        if model.best_iteration_ is not None
        else model.n_estimators
    )

    validation_predictions = model.predict(
        X_validation,
        num_iteration=best_iteration,
    )
    unseen_predictions = model.predict(
        X_unseen,
        num_iteration=best_iteration,
    )

    validation_mse = float(
        mean_squared_error(y_validation, validation_predictions)
    )
    unseen_pearson = safe_pearson(y_unseen, unseen_predictions)
    unseen_spearman = safe_spearman(y_unseen, unseen_predictions)

    trial.set_user_attr("best_iteration", int(best_iteration))
    trial.set_user_attr("unseen_pearson", unseen_pearson)
    trial.set_user_attr("unseen_spearman", unseen_spearman)

    trial_records.append(
        {
            "trial": trial.number,
            "validation_mse": validation_mse,
            "unseen_pearson": unseen_pearson,
            "unseen_spearman": unseen_spearman,
        }
    )

    # Retain the fitted model so the best trial can be saved directly.
    trial_models[trial.number] = model

    print(
        f"Trial {trial.number:3d} | "
        f"Validation MSE: {validation_mse:.6f} | "
        f"Unseen Pearson: {unseen_pearson:.4f} | "
        f"Unseen Spearman: {unseen_spearman:.4f}"
    )

    return validation_mse


sampler = optuna.samplers.TPESampler(seed=OPTUNA_SEED)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name="rs3_fixed_split_100_trials",
)

study.optimize(objective, n_trials=N_TRIALS)


[I 2026-07-11 17:19:42,260] A new study created in memory with name: rs3_fixed_split_100_trials
[I 2026-07-11 17:20:22,267] Trial 0 finished with value: 0.5649625605349683 and parameters: {'num_leaves': 101, 'min_child_samples': 244}. Best is trial 0 with value: 0.5649625605349683.


Trial   0 | Validation MSE: 0.564963 | Unseen Pearson: 0.6627 | Unseen Spearman: 0.6458


[I 2026-07-11 17:21:02,980] Trial 1 finished with value: 0.5674606755251624 and parameters: {'num_leaves': 190, 'min_child_samples': 157}. Best is trial 0 with value: 0.5649625605349683.


Trial   1 | Validation MSE: 0.567461 | Unseen Pearson: 0.6596 | Unseen Spearman: 0.6425


[I 2026-07-11 17:21:23,441] Trial 2 finished with value: 0.5801804998308076 and parameters: {'num_leaves': 46, 'min_child_samples': 46}. Best is trial 0 with value: 0.5649625605349683.


...

Trial  97 | Validation MSE: 0.564354 | Unseen Pearson: 0.6608 | Unseen Spearman: 0.6439


[I 2026-07-11 18:26:03,036] Trial 98 finished with value: 0.5654837916516794 and parameters: {'num_leaves': 207, 'min_child_samples': 153}. Best is trial 40 with value: 0.5614106886949427.


Trial  98 | Validation MSE: 0.565484 | Unseen Pearson: 0.6619 | Unseen Spearman: 0.6449


[I 2026-07-11 18:26:39,181] Trial 99 finished with value: 0.5632270238606174 and parameters: {'num_leaves': 109, 'min_child_samples': 161}. Best is trial 40 with value: 0.5614106886949427.


Trial  99 | Validation MSE: 0.563227 | Unseen Pearson: 0.6616 | Unseen Spearman: 0.6445


## Save the three requested metrics and the best model


In [19]:
results_df = (
    pd.DataFrame(trial_records)
    .sort_values("trial")
    .reset_index(drop=True)
)

# Combined file
results_df.to_csv(
    OUTPUT_DIR / "all_100_trial_metrics.csv",
    index=False,
)

# Numeric-only files
results_df["validation_mse"].to_csv(
    OUTPUT_DIR / "Validation_loss.txt",
    index=False,
    header=False,
)

results_df["unseen_pearson"].to_csv(
    OUTPUT_DIR / "Unseen_Pearson.txt",
    index=False,
    header=False,
)

results_df["unseen_spearman"].to_csv(
    OUTPUT_DIR / "Unseen_Spearman.txt",
    index=False,
    header=False,
)

best_trial_number = study.best_trial.number
best_row = results_df.loc[
    results_df["trial"] == best_trial_number
].iloc[0]

best_model = trial_models[best_trial_number]

joblib.dump(
    {
        "model": best_model,
        "feature_columns": X_train.columns.tolist(),
        "best_trial": best_trial_number,
        "best_params": study.best_trial.params,
        "best_iteration": study.best_trial.user_attrs["best_iteration"],
        "split_seed": SPLIT_SEED,
        "validation_mse": float(best_row["validation_mse"]),
        "unseen_pearson": float(best_row["unseen_pearson"]),
        "unseen_spearman": float(best_row["unseen_spearman"]),
    },
    OUTPUT_DIR / "best_model.joblib",
)

with open(OUTPUT_DIR / "best_trial_summary.json", "w") as f:
    json.dump(
        {
            "best_trial": int(best_trial_number),
            "best_params": study.best_trial.params,
            "best_iteration": int(
                study.best_trial.user_attrs["best_iteration"]
            ),
            "validation_mse": float(best_row["validation_mse"]),
            "unseen_pearson": float(best_row["unseen_pearson"]),
            "unseen_spearman": float(best_row["unseen_spearman"]),
        },
        f,
        indent=2,
    )

display(results_df.head())

print("\nBest trial selected by validation MSE")
print("Trial:", best_trial_number)
print("Parameters:", study.best_trial.params)
print(f"Validation MSE: {best_row['validation_mse']:.6f}")
print(f"Unseen Pearson: {best_row['unseen_pearson']:.4f}")
print(f"Unseen Spearman: {best_row['unseen_spearman']:.4f}")
print("\nFiles saved in:", OUTPUT_DIR.resolve())


,trial,validation_mse,unseen_pearson,unseen_spearman
0,0,0.564963,0.662682,0.645836
1,1,0.567461,0.659578,0.642520
2,2,0.580180,0.652691,0.634932
3,3,0.570332,0.655088,0.638557
4,4,0.566825,0.658645,0.641800



Best trial selected by validation MSE
Trial: 40
Parameters: {'num_leaves': 125, 'min_child_samples': 122}
Validation MSE: 0.561411
Unseen Pearson: 0.6619
Unseen Spearman: 0.6439

Files saved in: /Users/zhangjiongyu/rs_dev/models/rs3_fixed_split_100_trials


## Optional: verify output files


In [21]:
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)


Unseen_Pearson.txt
Unseen_Spearman.txt
Validation_loss.txt
all_100_trial_metrics.csv
best_model.joblib
best_trial_summary.json
